# Fabric Tenant-Wide Dataset Refresh Audit

This notebook scans all workspaces in your Microsoft Fabric tenant and retrieves the refresh configuration (Full / Incremental / DirectQuery) for every dataset (semantic model).

It is designed to run **directly inside a Microsoft Fabric notebook**.

## Prerequisites
- Run this notebook with an identity that has **Power BI / Fabric Admin** rights (Fabric Administrator role, or a service principal with `Tenant.Read.All` Admin API access).
- Attaching a **Lakehouse** is optional but recommended if you want to persist the results as a Delta table.
- The default authentication uses the notebook's own identity via `notebookutils.credentials.getToken`, so no secrets are required. A service principal fallback is included for automated jobs.

## 0. Parameters

This cell is tagged as **parameters** so it can be overridden when the notebook is run from a Fabric pipeline or scheduled job.

In [8]:
# Authentication method: 'fabric_integrated' (default, uses notebook identity),
# 'service_principal', or 'default_azure'
auth_method = "fabric_integrated"

# Only required when auth_method == 'service_principal'
tenant_id = ""
client_id = ""
client_secret = ""

# Number of parallel workers for enriching dataset details (mind API rate limits)
max_workers = 5

# Set to True to save results to an attached Lakehouse as a Delta table
save_to_lakehouse = True
lakehouse_table_name = "dataset_refresh_audit"

StatementMeta(, 0e18a0f3-de53-4bb5-b985-fecdd00934d6, 10, Finished, Available, Finished, False)

## 1. Imports

In [9]:
import os
import time
import json
import logging
import requests
import pandas as pd
from typing import Dict, List, Optional
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

StatementMeta(, 0e18a0f3-de53-4bb5-b985-fecdd00934d6, 11, Finished, Available, Finished, False)

## 2. Authentication

In Fabric, the simplest and most secure option is `fabric_integrated`, which uses the notebook's own identity via `notebookutils.credentials.getToken`. No secrets are stored in the notebook.

For automated (non-interactive) runs outside Fabric, a **service principal** or **DefaultAzureCredential** can be used instead.

In [10]:
class FabricAuthenticator:
    """Handles authentication to the Power BI Admin API."""

    # Audience for Fabric / Power BI REST APIs
    PBI_RESOURCE = "https://analysis.windows.net/powerbi/api"
    PBI_SCOPE = "https://analysis.windows.net/powerbi/api/.default"

    def __init__(
        self,
        tenant_id: Optional[str] = None,
        client_id: Optional[str] = None,
        client_secret: Optional[str] = None,
        auth_method: str = "fabric_integrated",
    ):
        self.tenant_id = tenant_id or os.getenv('AZURE_TENANT_ID')
        self.client_id = client_id or os.getenv('AZURE_CLIENT_ID')
        self.client_secret = client_secret or os.getenv('AZURE_CLIENT_SECRET')
        self.auth_method = auth_method

    def get_access_token(self) -> str:
        """Get an access token for the Power BI Admin API."""
        try:
            if self.auth_method == "fabric_integrated":
                # Uses the notebook's own identity - no secrets required.
                import notebookutils  # available in the Fabric runtime
                token = notebookutils.credentials.getToken(self.PBI_RESOURCE)
                logger.info("Authenticated using Fabric notebook identity")
                return token

            if self.auth_method == "service_principal":
                from azure.identity import ClientSecretCredential
                if not all([self.tenant_id, self.client_id, self.client_secret]):
                    raise ValueError(
                        "Service Principal auth requires tenant_id, client_id and client_secret "
                        "(parameters or AZURE_TENANT_ID / AZURE_CLIENT_ID / AZURE_CLIENT_SECRET env vars)."
                    )
                credential = ClientSecretCredential(
                    tenant_id=self.tenant_id,
                    client_id=self.client_id,
                    client_secret=self.client_secret,
                )
                logger.info("Authenticated using service principal")
                return credential.get_token(self.PBI_SCOPE).token

            # DefaultAzureCredential (CLI, managed identity, env, etc.)
            from azure.identity import DefaultAzureCredential
            credential = DefaultAzureCredential()
            logger.info("Authenticated using DefaultAzureCredential")
            return credential.get_token(self.PBI_SCOPE).token

        except Exception as e:
            logger.error(f"Authentication failed: {e}")
            raise

StatementMeta(, 0e18a0f3-de53-4bb5-b985-fecdd00934d6, 12, Finished, Available, Finished, False)

## 3. Power BI Admin API Client

Handles all interactions with the Power BI Admin API endpoints, including pagination, retries and rate-limit handling.

| Endpoint | Method | Purpose |
|----------|--------|---------|
| `/admin/workspaces` | GET | List all workspaces in tenant |
| `/admin/datasets` | GET | List all datasets across tenant |
| `/admin/datasets/{id}` | GET | Get detailed dataset configuration |

In [11]:
class PowerBIAdminAPIClient:
    """Client for interacting with the Power BI Admin API."""

    ADMIN_BASE_URL = "https://api.powerbi.com/v1.0/myorg/admin"

    def __init__(self, authenticator: FabricAuthenticator):
        self.authenticator = authenticator
        self.session = requests.Session()
        self.headers = {}
        self._refresh_headers()

    def _refresh_headers(self):
        token = self.authenticator.get_access_token()
        self.headers = {
            'Authorization': f'Bearer {token}',
            'Content-Type': 'application/json',
        }

    def _make_request(
        self,
        method: str,
        endpoint: str,
        params: Optional[Dict] = None,
        data: Optional[Dict] = None,
        retry_count: int = 3,
    ) -> Dict:
        url = f"{self.ADMIN_BASE_URL}{endpoint}"

        for attempt in range(retry_count):
            try:
                response = self.session.request(
                    method=method,
                    url=url,
                    headers=self.headers,
                    params=params,
                    json=data,
                    timeout=30,
                )

                # Handle rate limiting
                if response.status_code == 429:
                    retry_after = int(response.headers.get('Retry-After', 60))
                    logger.warning(f"Rate limited. Waiting {retry_after}s...")
                    time.sleep(retry_after)
                    continue

                response.raise_for_status()
                return response.json() if response.text else {}

            except requests.exceptions.RequestException as e:
                if attempt < retry_count - 1:
                    wait_time = 2 ** attempt  # Exponential backoff
                    logger.warning(
                        f"Request failed (attempt {attempt + 1}/{retry_count}): {e}. "
                        f"Retrying in {wait_time}s..."
                    )
                    time.sleep(wait_time)
                else:
                    logger.error(f"Request failed after {retry_count} attempts: {e}")
                    raise
        return {}

    def get_workspaces(self) -> List[Dict]:
        """Get all workspaces in the tenant (paginated)."""
        logger.info("Fetching all workspaces...")
        all_workspaces, skip, top = [], 0, 100
        while True:
            response = self._make_request(
                method='GET', endpoint='/workspaces',
                params={'$skip': skip, '$top': top},
            )
            workspaces = response.get('value', [])
            if not workspaces:
                break
            all_workspaces.extend(workspaces)
            logger.info(f"Fetched {len(workspaces)} workspaces (total: {len(all_workspaces)})")
            skip += top
        logger.info(f"Total workspaces found: {len(all_workspaces)}")
        return all_workspaces

    def get_datasets(self) -> List[Dict]:
        """Get all datasets across the entire tenant (paginated)."""
        logger.info("Fetching all datasets across tenant...")
        all_datasets, skip, top = [], 0, 100
        while True:
            response = self._make_request(
                method='GET', endpoint='/datasets',
                params={'$skip': skip, '$top': top},
            )
            datasets = response.get('value', [])
            if not datasets:
                break
            all_datasets.extend(datasets)
            logger.info(f"Fetched {len(datasets)} datasets (total: {len(all_datasets)})")
            skip += top
        logger.info(f"Total datasets found: {len(all_datasets)}")
        return all_datasets

    def get_dataset_details(self, dataset_id: str) -> Dict:
        """Get detailed information about a specific dataset."""
        try:
            return self._make_request(method='GET', endpoint=f'/datasets/{dataset_id}')
        except Exception as e:
            logger.error(f"Error fetching dataset {dataset_id} details: {e}")
            return {}

StatementMeta(, 0e18a0f3-de53-4bb5-b985-fecdd00934d6, 13, Finished, Available, Finished, False)

## 4. Refresh Configuration Parser

Extracts and interprets refresh type configurations from dataset metadata.

| Refresh Type | Meaning | Storage Mode |
|--------------|---------|--------------|
| Full Refresh | Complete data reload each cycle | Import |
| Incremental Refresh | Only new/changed data loaded | Import |
| DirectQuery (No Refresh) | Live connection | DirectQuery |
| No Schedule | Configured but not scheduled | Import |
| Not Configured | No refresh settings | Unknown |

In [12]:
class RefreshConfigParser:
    """Parse and interpret refresh configurations from dataset metadata."""

    @staticmethod
    def extract_refresh_config(dataset: Dict) -> Dict:
        refresh_config = {
            'refresh_type': 'Not Configured',
            'is_incremental': False,
            'refresh_schedule_enabled': False,
            'scheduled_refresh_times': [],
            'raw_config': {},
        }

        schedule = dataset.get('refreshSchedule')
        if schedule:
            refresh_config['refresh_schedule_enabled'] = True
            if 'frequency' in schedule:
                refresh_config['refresh_type'] = 'Full Refresh'
            if 'times' in schedule:
                refresh_config['scheduled_refresh_times'] = schedule.get('times', [])

        default_mode = dataset.get('defaultMode')
        if default_mode == 'DirectQuery':
            refresh_config['refresh_type'] = 'DirectQuery (No Refresh)'
            refresh_config['is_incremental'] = False
        elif default_mode == 'Import':
            if refresh_config['refresh_schedule_enabled']:
                refresh_config['refresh_type'] = 'Full Refresh'
            else:
                refresh_config['refresh_type'] = 'No Schedule'

        refresh_config['raw_config'] = dataset.get('refreshSchedule', {})
        return refresh_config

StatementMeta(, 0e18a0f3-de53-4bb5-b985-fecdd00934d6, 14, Finished, Available, Finished, False)

## 5. Tenant Audit Engine

Orchestrates scanning of all workspaces and datasets, then builds a structured results table.

In [13]:
class TenantDatasetAudit:
    """Main audit engine for tenant-wide dataset discovery and refresh analysis."""

    def __init__(self, client: PowerBIAdminAPIClient):
        self.client = client
        self.workspaces = {}
        self.datasets = []

    def run_audit(self, max_workers: int = 5) -> pd.DataFrame:
        logger.info("Starting tenant-wide dataset audit...")
        start_time = datetime.now()

        self._fetch_workspaces()
        self._fetch_all_datasets()
        self._enrich_datasets(max_workers=max_workers)
        results_df = self._build_results_dataframe()

        elapsed = (datetime.now() - start_time).total_seconds()
        logger.info(
            f"Audit completed in {elapsed:.2f}s. "
            f"Found {len(self.workspaces)} workspaces, {len(results_df)} datasets."
        )
        return results_df

    def _fetch_workspaces(self):
        logger.info("Step 1: Fetching workspaces...")
        for workspace in self.client.get_workspaces():
            self.workspaces[workspace['id']] = {
                'id': workspace['id'],
                'name': workspace.get('name', 'Unknown'),
                'type': workspace.get('type', 'Unknown'),
                'state': workspace.get('state', 'Unknown'),
            }
        logger.info(f"Cached {len(self.workspaces)} workspaces")

    def _fetch_all_datasets(self):
        logger.info("Step 2: Fetching all datasets...")
        self.datasets = self.client.get_datasets()
        logger.info(f"Found {len(self.datasets)} datasets")

    def _enrich_datasets(self, max_workers: int = 5):
        logger.info(f"Step 3: Enriching datasets (max_workers={max_workers})...")
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {
                executor.submit(self.client.get_dataset_details, ds['id']): ds
                for ds in self.datasets
            }
            completed = 0
            for future in as_completed(futures):
                try:
                    details = future.result()
                    if details:
                        futures[future].update(details)
                    completed += 1
                    if completed % 100 == 0:
                        logger.info(f"Enriched {completed}/{len(self.datasets)} datasets")
                except Exception as e:
                    logger.warning(f"Failed to enrich dataset: {e}")
        logger.info(f"Completed enrichment of {len(self.datasets)} datasets")

    def _build_results_dataframe(self) -> pd.DataFrame:
        logger.info("Building results dataframe...")
        results = []
        for dataset in self.datasets:
            workspace_id = dataset.get('workspaceId', '')
            workspace = self.workspaces.get(workspace_id, {})
            refresh_config = RefreshConfigParser.extract_refresh_config(dataset)
            results.append({
                'Workspace Name': workspace.get('name', 'Unknown'),
                'Workspace ID': workspace_id,
                'Dataset Name': dataset.get('name', 'Unknown'),
                'Dataset ID': dataset.get('id', ''),
                'Refresh Type': refresh_config['refresh_type'],
                'Is Incremental': refresh_config['is_incremental'],
                'Schedule Enabled': refresh_config['refresh_schedule_enabled'],
                'Scheduled Times': ', '.join(refresh_config['scheduled_refresh_times'])
                    if refresh_config['scheduled_refresh_times'] else 'None',
                'Default Mode': dataset.get('defaultMode', 'Unknown'),
                'Owner': dataset.get('configuredBy', dataset.get('owner', 'Unknown')),
                'Target Storage Mode': dataset.get('targetStorageMode', 'Unknown'),
            })
        df = pd.DataFrame(results)
        if not df.empty:
            df = df.sort_values(['Workspace Name', 'Dataset Name']).reset_index(drop=True)
        logger.info(f"Results dataframe built: {len(df)} rows")
        return df

StatementMeta(, 0e18a0f3-de53-4bb5-b985-fecdd00934d6, 15, Finished, Available, Finished, False)

## 6. Run the Audit

This uses the parameters defined at the top of the notebook.

In [14]:
logger.info("=" * 80)
logger.info("FABRIC TENANT-WIDE DATASET REFRESH AUDIT")
logger.info("=" * 80)

# Temporary compatibility patch:
# The correct Power BI Admin API endpoint for listing workspaces is
# /admin/groups ("GetGroupsAsAdmin"), not /admin/workspaces.
# The original PowerBIAdminAPIClient was using '/workspaces', which
# returns HTTP 404. Here we monkey‑patch the client's get_workspaces
# method to call the correct endpoint with the same pagination logic.

from types import MethodType

def _patched_get_workspaces(self):
    """Get all workspaces in the tenant (paginated) using /admin/groups."""
    logger.info("Fetching all workspaces (via /admin/groups)...")
    all_workspaces, skip, top = [], 0, 100
    while True:
        response = self._make_request(
            method='GET',
            endpoint='/groups',  # Correct admin API endpoint
            params={'$skip': skip, '$top': top},
        )
        workspaces = response.get('value', [])
        if not workspaces:
            break
        all_workspaces.extend(workspaces)
        logger.info(
            f"Fetched {len(workspaces)} workspaces (total: {len(all_workspaces)})"
        )
        skip += top
    logger.info(f"Total workspaces found: {len(all_workspaces)}")
    return all_workspaces

authenticator = FabricAuthenticator(
    tenant_id=tenant_id or None,
    client_id=client_id or None,
    client_secret=client_secret or None,
    auth_method=auth_method,
)

client = PowerBIAdminAPIClient(authenticator)
# Apply the patch so that TenantDatasetAudit uses the corrected method
client.get_workspaces = MethodType(_patched_get_workspaces, client)

audit = TenantDatasetAudit(client)
results_df = audit.run_audit(max_workers=max_workers)

print(f"\nTotal Datasets:   {len(results_df)}")
if not results_df.empty:
    print(f"Total Workspaces: {results_df['Workspace ID'].nunique()}")
    print("\nRefresh Type Distribution:")
    print(results_df['Refresh Type'].value_counts())
    print("\nDataset Mode Distribution:")
    print(results_df['Default Mode'].value_counts())
    print(f"\nSchedule Enabled: {results_df['Schedule Enabled'].sum()} datasets")

StatementMeta(, 0e18a0f3-de53-4bb5-b985-fecdd00934d6, 16, Finished, Available, Finished, False)

2026-07-02 02:19:55,590 - INFO - ================================================================================
2026-07-02 02:19:55,591 - INFO - FABRIC TENANT-WIDE DATASET REFRESH AUDIT
2026-07-02 02:19:55,592 - INFO - ================================================================================
2026-07-02 02:19:55,597 - INFO - Authenticated using Fabric notebook identity
2026-07-02 02:19:55,598 - INFO - Starting tenant-wide dataset audit...
2026-07-02 02:19:55,598 - INFO - Step 1: Fetching workspaces...
2026-07-02 02:19:55,599 - INFO - Fetching all workspaces (via /admin/groups)...
2026-07-02 02:19:55,888 - INFO - Fetched 65 workspaces (total: 65)
2026-07-02 02:19:55,961 - INFO - Total workspaces found: 65
2026-07-02 02:19:55,962 - INFO - Cached 65 workspaces
2026-07-02 02:19:55,962 - INFO - Step 2: Fetching all datasets...
2026-07-02 02:19:55,963 - INFO - Fetching all datasets across tenant...
2026-07-02 02:19:56,138 - INFO - Fetched 66 datasets (total: 66)
2026-07-02 02:19:56,2


Total Datasets:   66
Total Workspaces: 29

Refresh Type Distribution:
Refresh Type
Not Configured    66
Name: count, dtype: int64

Dataset Mode Distribution:
Default Mode
Unknown    66
Name: count, dtype: int64

Schedule Enabled: 0 datasets


In [15]:
# Display the full results table (rich table view in Fabric)
display(results_df)

StatementMeta(, 0e18a0f3-de53-4bb5-b985-fecdd00934d6, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 81d986ea-7618-44f4-b1ad-da675e26cdfd)

## 7. Save Results to Lakehouse (optional)

Set `save_to_lakehouse = True` in the parameters cell and attach a Lakehouse to persist the audit as a Delta table.

In [17]:
if save_to_lakehouse and not results_df.empty:
    import re

    # Add an audit timestamp column
    export_df = results_df.copy()
    export_df['Audit Timestamp'] = datetime.utcnow().isoformat()

    # Sanitize column names for Delta: replace spaces and other invalid chars with '_'
    invalid_chars_pattern = r"[ ,;{}()\n\t=]"
    export_df.columns = [re.sub(invalid_chars_pattern, "_", str(c)) for c in export_df.columns]

    # Use the Fabric-provided Spark session (do NOT create a new one)
    spark_df = spark.createDataFrame(export_df)
    (spark_df.write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .saveAsTable(lakehouse_table_name))

    print(f"✓ Results saved to Lakehouse table: {lakehouse_table_name}")
else:
    print("Skipping Lakehouse save (save_to_lakehouse is False or no results).")

StatementMeta(, 0e18a0f3-de53-4bb5-b985-fecdd00934d6, 19, Finished, Available, Finished, False)

✓ Results saved to Lakehouse table: dataset_refresh_audit


In [19]:
df = spark.sql("SELECT * FROM Fabric_Audit_LH.dataset_refresh_audit LIMIT 1000")
display(df)

StatementMeta(, 0e18a0f3-de53-4bb5-b985-fecdd00934d6, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ac6636db-5bd1-45e5-baa6-b63c4b1f955f)